# Qwen3.8-27B on Colab: paired context audit

Run the existing four-condition experiment on one Colab **H100** with the pinned
`Qwen/Qwen3.8-27B` model. The monitor and summarizer use independent text-only
requests; transcript commands are inert data. Thinking is disabled in the backend.

**Default Run All is inert:** no installation, Drive mounting, download, GPU access,
data acquisition, or model generation. Edit the flags below only when ready for
that phase. Real pilot performance, memory fit and runtime still need validation.

Work in this order: **setup → acquisition → three-pair pilot → inspect development
outputs → all development pairs → reviewed freeze → test → offline analysis**.
The test requires its own opt-in. Local source, model weights and the server live
in `/content`; data, run records, source provenance and pins persist in your Drive.
Never share the private Drive directory or notebook outputs containing source text.

In [ ]:
from pathlib import Path

# Explicit notebook edits, never environment-variable opt-ins.
RUN_SETUP = False
RUN_ACQUIRE = False
RUN_LIVE = False
RUN_FREEZE = False
REVIEWED_FREEZE = False
RUN_TEST = False
RUN_ANALYSIS = False
RUN_RECONCILE = False
CONFIRM_RECONCILIATION = False
RECONCILE_SESSION_ID = ""
RECONCILE_ELAPSED_SECONDS = None
DISCONNECT_AFTER_RUN = True
PHASE = "pilot"  # "pilot", "development", or "test"

MODEL_ID = "Qwen/Qwen3.8-27B"
MODEL_REVISION = ""  # Exact 40-character HF commit; first setup resolves and saves it once.
VLLM_VERSION = "0.28.0"
MAX_MODEL_LEN = 65536  # Validate full-input feasibility during the development pilot.
GPU_HOURLY_RATE_USD = None  # Your explicit effective hourly rate, not a guessed Colab price.
MAX_COST_USD = None  # Explicit cumulative cap for this phase/run directory.
SESSION_MAX_SECONDS = 3600
STARTUP_TIMEOUT_SECONDS = 900
DATA_USE_CONFIRMED = False
RUBRIC_REVIEWED = False

REPO = Path("/content/agent-monitor-context-audit")
DRIVE_ROOT = Path("/content/drive/MyDrive/agent-monitor-context-audit-private")
REPO_URL = "https://github.com/gustavogomespl/agent-monitor-context-audit.git"
BRANCH = "pilot"  # Published branch with Qwen code; change before the first setup.
CODE_REF = ""  # Optional exact 40-character commit; blank resolves BRANCH once.
PROJECT_ZIP = ""  # Optional bundle override; clear REPO_URL to use the upload picker.
SETUP_READY = False
print("No setup or generation is enabled by default. Choose one explicit phase when ready.")

## 1. Explicit setup and durable source

Select `BRANCH` (default **pilot**) in this repository, then set `RUN_SETUP=True`.
Setup clones/fetches that branch, checks out its exact commit, installs dependencies
and prints the source commit and Python version. No ZIP is required. `CODE_REF` is
an optional exact commit override. The selected SHA is saved in Drive at
`configuration/code-pin.json`; reconnects reuse it even when the branch advances.
Use a new Drive workspace and fresh runtime to intentionally select different code.

For unpublished source, set `PROJECT_ZIP` to the supplied **qwen-colab-bundle.zip**,
or clear `REPO_URL` to use the upload picker. A saved ZIP workspace keeps its source.
Reviewed frozen snapshots always take precedence over the remote branch.

Set `RUN_SETUP=True` and run the setup cell. It mounts Drive, restores your saved
source when available, installs the pinned vLLM version and project dependencies,
and saves model/runtime provenance. The first setup resolves an empty model revision
through Hugging Face's read-only metadata API; later setups reuse the saved SHA.
A changed model or engine pin requires a new workspace and exploratory version.

The managed financial cap starts with the live helper, **not while Colab is idle,
installing packages, or waiting for you**. Use a CPU runtime for preparation when
possible; select H100 for the live phase and rerun setup after a runtime change.
Dependencies and weights can take substantial time and disk space. The exact versions of vLLM, PyTorch, Transformers, Tokenizers, Triton and Safetensors
are saved in the durable model pin and reused on reconnect. Transformers must satisfy
`>=5.8.0,<6`. Setup and the live supervisor reject any package drift: reinstall the
saved exact versions or start a documented exploratory workspace. Actual H100
validation belongs to the pilot.

In [ ]:
def bind_git_metadata(repo, durable_git, expected_commit=None):
    """Keep restored source files and their durable Git HEAD consistent."""
    import shutil
    import subprocess

    if expected_commit is not None and durable_git.exists():
        durable_head = subprocess.run(
            ["git", f"--git-dir={durable_git}", "rev-parse", "HEAD"],
            capture_output=True, text=True, check=True,
        ).stdout.strip()
        if durable_head != expected_commit:
            raise ValueError(
                "Durable Git HEAD differs from the source commit; restore its matching snapshot."
            )
    local_git = repo / ".git"
    if local_git.is_symlink():
        if local_git.resolve() != durable_git.resolve():
            raise ValueError("Local source uses a different persistent Git workspace.")
    else:
        if durable_git.exists():
            if local_git.exists():
                shutil.rmtree(local_git)
        elif local_git.is_dir():
            shutil.copytree(local_git, durable_git)
            shutil.rmtree(local_git)
        else:
            raise ValueError("Source Git metadata is missing.")
        local_git.symlink_to(durable_git, target_is_directory=True)


def checkout_source(repo, repo_url, branch, code_ref, pin_path):
    """Select a branch once; preserve the exact source and local work on reconnect."""
    import json
    import re
    import subprocess

    if not repo_url or not branch or (code_ref and not re.fullmatch(r"[0-9a-f]{40}", code_ref)):
        raise ValueError("Provide a repository, a branch and an optional exact 40-character SHA.")
    valid = subprocess.run(
        ["git", "check-ref-format", "--branch", branch], capture_output=True, text=True
    )
    if valid.returncode:
        raise ValueError("Invalid source branch name.")
    saved = json.loads(pin_path.read_text()) if pin_path.exists() else None
    if saved is not None:
        if (
            saved.get("repo_url") != repo_url
            or saved.get("branch") != branch
            or not re.fullmatch(r"[0-9a-f]{40}", saved.get("commit", ""))
            or (code_ref and code_ref != saved["commit"])
        ):
            raise ValueError("Source selection differs from the saved pin; use a new workspace.")
    if repo.exists():
        if saved is None or not (repo / ".git").exists():
            raise ValueError("Existing checkout lacks a source pin; preserve it first.")
        head = subprocess.run(
            ["git", "rev-parse", "HEAD"], cwd=repo, capture_output=True, text=True, check=True
        ).stdout.strip()
        if head != saved["commit"]:
            raise ValueError("Existing HEAD differs from the source pin; no checkout was changed.")
        # Inventory manifests may be modified locally. Never reset or pull over them.
        return saved

    subprocess.run(["git", "clone", "--no-checkout", "--", repo_url, str(repo)], check=True)
    target = saved["commit"] if saved else code_ref or f"refs/heads/{branch}"
    subprocess.run(["git", "fetch", "origin", target], cwd=repo, check=True)
    commit = subprocess.run(
        ["git", "rev-parse", "FETCH_HEAD^{commit}"],
        cwd=repo, capture_output=True, text=True, check=True,
    ).stdout.strip()
    if not re.fullmatch(r"[0-9a-f]{40}", commit) or (saved and commit != saved["commit"]):
        raise ValueError("Fetched source does not match its immutable commit.")
    subprocess.run(["git", "checkout", "--detach", commit], cwd=repo, check=True)
    if saved is None:
        saved = {"repo_url": repo_url, "branch": branch, "commit": commit}
        pin_path.parent.mkdir(parents=True, exist_ok=True)
        temporary = pin_path.with_suffix(".tmp")
        temporary.write_text(json.dumps(saved, indent=2) + "\n")
        temporary.replace(pin_path)
    return saved


def require_setup():
    if not SETUP_READY:
        raise RuntimeError("Run the explicit RUN_SETUP phase in this runtime first.")


def safe_extract(archive_path, target):
    import stat
    import zipfile

    target = target.resolve()
    with zipfile.ZipFile(archive_path) as archive:
        for item in archive.infolist():
            destination = (target / item.filename).resolve()
            if not destination.is_relative_to(target) or Path(item.filename).is_absolute():
                raise ValueError("Unsafe archive path; extraction refused.")
            if ".git" in Path(item.filename).parts or stat.S_ISLNK(item.external_attr >> 16):
                raise ValueError("Archive must contain ordinary source files, not Git metadata.")
        archive.extractall(target)


def bind_private_directory(relative, durable):
    import shutil

    link = REPO / relative
    durable.mkdir(parents=True, exist_ok=True)
    link.parent.mkdir(parents=True, exist_ok=True)
    if link.is_symlink():
        if link.resolve() != durable.resolve():
            raise ValueError("Existing private link points to another workspace.")
        return
    if link.exists():
        if any(link.iterdir()):
            raise ValueError("Existing local private data requires an explicit migration first.")
        shutil.rmtree(link)
    link.symlink_to(durable, target_is_directory=True)


def save_public_manifests():
    import shutil

    destination = DRIVE_ROOT / "public-manifests"
    destination.mkdir(parents=True, exist_ok=True)
    for source in (REPO / "data/manifests").glob("*"):
        if source.is_file():
            shutil.copy2(source, destination / source.name)


def verify_runtime_versions(pin_path, installed):
    import json

    pin = json.loads(pin_path.read_text())
    expected = pin.get("runtime_versions")
    if expected is not None and expected != installed:
        raise RuntimeError(
            "Inference dependencies differ from the saved runtime pin. Reinstall the exact "
            "pinned versions, or use a new explicitly exploratory workspace."
        )
    if expected is None:
        pin["runtime_versions"] = dict(installed)
        pin_path.write_text(json.dumps(pin, indent=2) + "\n")
    return pin


def configured_phase(phase):
    import json

    from context_audit.runtime_models import AuditConfig, QwenConfig

    require_setup()
    if phase not in {"pilot", "development", "test"}:
        raise ValueError("Choose pilot, development, or test.")
    if not DATA_USE_CONFIRMED or not RUBRIC_REVIEWED:
        raise ValueError("Confirm data-use compatibility and review the rubric before generation.")
    pin = json.loads((DRIVE_ROOT / "configuration/model-pin.json").read_text())
    backend = QwenConfig(
        model_revision=pin["model_revision"],
        vllm_version=pin["vllm_version"],
        runtime_versions=pin["runtime_versions"],
        base_url="http://127.0.0.1:8000",
        max_model_len=MAX_MODEL_LEN,
        gpu_hourly_rate_usd=GPU_HOURLY_RATE_USD,
    )
    backend.validate_live()
    config = AuditConfig(
        provider="qwen_local",
        qwen=backend,
        protocol_version="protocol-v1" if phase == "test" else "development-v1",
        split="test" if phase == "test" else "development",
        dataset_dir="data/private",
        run_dir=f"runs/private/qwen-{phase}",
        monitor_model=pin["model_id"],
        summarizer_model=pin["model_id"],
        monitor_context_window=MAX_MODEL_LEN,
        summarizer_context_window=MAX_MODEL_LEN,
        pilot_pairs=3 if phase == "pilot" else None,
        timeout_seconds=300,
        data_use_confirmed=DATA_USE_CONFIRMED,
        rubric_reviewed=RUBRIC_REVIEWED,
        protocol_file="data/manifests/protocol-v1.json",
    )
    path = DRIVE_ROOT / "configuration" / f"{phase}.json"
    payload = config.model_dump()
    if path.exists() and json.loads(path.read_text()) != payload:
        raise ValueError(
            "This phase already has a different saved configuration. Preserve its results and "
            "use a new explicit workspace/version for changed methods."
        )
    if not path.exists():
        path.write_text(json.dumps(payload, indent=2) + "\n")
    return config

In [ ]:
if RUN_SETUP:
    import importlib.metadata
    import json
    import re
    import shutil
    import subprocess
    import sys
    import tempfile
    import urllib.parse
    import urllib.request
    from datetime import datetime, timezone

    from google.colab import drive, files

    drive.mount("/content/drive")
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
    configuration = DRIVE_ROOT / "configuration"
    configuration.mkdir(exist_ok=True)
    saved_upload = DRIVE_ROOT / "source-upload.zip"
    frozen_source = DRIVE_ROOT / "frozen-source.zip"
    durable_git = DRIVE_ROOT / "git-metadata"
    code_pin_path = configuration / "code-pin.json"
    source_kind = "git" if REPO_URL and not PROJECT_ZIP and not saved_upload.exists() else "bundle"
    if frozen_source.exists():
        source_kind = "frozen"
        if not durable_git.exists():
            raise ValueError("Frozen source lacks durable Git provenance; restore it first.")
        if not REPO.exists():
            REPO.mkdir(parents=True)
            safe_extract(frozen_source, REPO)
    elif source_kind == "git":
        checkout_source(REPO, REPO_URL, BRANCH, CODE_REF, code_pin_path)
    elif not REPO.exists():
        if not saved_upload.exists():
            if PROJECT_ZIP:
                source_zip = Path(PROJECT_ZIP)
            else:
                uploaded = files.upload()
                if len(uploaded) != 1:
                    raise ValueError("Upload exactly one qwen-colab-bundle.zip.")
                source_zip = Path(next(iter(uploaded)))
            shutil.copy2(source_zip, saved_upload)
        with tempfile.TemporaryDirectory(prefix="qwen-source-") as staging:
            stage = Path(staging)
            safe_extract(saved_upload, stage)
            bundle = stage / "source.bundle"
            if not bundle.is_file() or not (stage / "research_plan.md").is_file():
                raise ValueError("Expected source.bundle and project files at the ZIP root.")
            subprocess.run(["git", "clone", str(bundle), str(REPO)], check=True)
            for source in stage.iterdir():
                if source.name == "source.bundle":
                    continue
                destination = REPO / source.name
                if source.is_dir():
                    shutil.copytree(source, destination, dirs_exist_ok=True)
                else:
                    shutil.copy2(source, destination)
    if not (REPO / "src/context_audit/colab.py").is_file():
        raise ValueError("This source revision does not include the Qwen Colab implementation.")
    expected_commit = (
        json.loads(code_pin_path.read_text())["commit"] if source_kind == "git" else None
    )
    bind_git_metadata(REPO, durable_git, expected_commit)
    bind_private_directory("data/private", DRIVE_ROOT / "data-private")
    bind_private_directory("runs/private", DRIVE_ROOT / "runs-private")
    saved_manifests = DRIVE_ROOT / "public-manifests"
    if saved_manifests.exists():
        shutil.copytree(saved_manifests, REPO / "data/manifests", dirs_exist_ok=True)

    pin_path = configuration / "model-pin.json"
    if pin_path.exists():
        pin = json.loads(pin_path.read_text())
        if pin["model_id"] != MODEL_ID or pin["vllm_version"] != VLLM_VERSION:
            raise ValueError("Model/engine differs from the durable pin; do not overwrite it.")
        if MODEL_REVISION and MODEL_REVISION != pin["model_revision"]:
            raise ValueError("Explicit model revision disagrees with the saved pin.")
    else:
        revision = MODEL_REVISION
        if not revision:
            model_path = urllib.parse.quote(MODEL_ID, safe="/")
            request = urllib.request.Request(
                f"https://huggingface.co/api/models/{model_path}/revision/main",
                headers={"User-Agent": "context-audit-colab-setup"},
            )
            with urllib.request.urlopen(request, timeout=30) as response:
                revision = json.load(response)["sha"]
        if not re.fullmatch(r"[0-9a-f]{40}", revision):
            raise ValueError("Model revision must be an exact immutable 40-character commit.")
        pin = {
            "model_id": MODEL_ID,
            "model_revision": revision,
            "vllm_version": VLLM_VERSION,
            "resolved_at": datetime.now(timezone.utc).isoformat(),
        }
        pin_path.write_text(json.dumps(pin, indent=2) + "\n")
    dependencies = [f"vllm=={pin['vllm_version']}", "transformers>=5.8.0,<6"]
    if "runtime_versions" in pin:
        dependencies.extend(
            f"{name}=={version}" for name, version in pin["runtime_versions"].items()
        )
    subprocess.run([sys.executable, "-m", "pip", "install", *dependencies], check=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-e", ".[data]"], cwd=REPO, check=True
    )
    inference_packages = ("vllm", "torch", "transformers", "tokenizers", "triton", "safetensors")
    installed_versions = {name: importlib.metadata.version(name) for name in inference_packages}
    pin = verify_runtime_versions(pin_path, installed_versions)
    # CPU-visible provenance only; setup neither starts an engine nor queries a GPU.
    packages = subprocess.run(
        [sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True, check=True
    ).stdout
    setup_history = configuration / "setup-history"
    setup_history.mkdir(exist_ok=True)
    timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    code_commit = subprocess.run(
        ["git", "rev-parse", "HEAD"], cwd=REPO, capture_output=True, text=True, check=True
    ).stdout.strip()
    setup_record = {
        "schema_version": 1,
        "created_at": timestamp,
        "python": sys.version,
        "code_commit": code_commit,
        "source_kind": source_kind,
        "source_pin": json.loads(code_pin_path.read_text()) if code_pin_path.exists() else None,
        "model_pin": pin,
        "installed_packages": packages.splitlines(),
    }
    (setup_history / f"{timestamp}.json").write_text(json.dumps(setup_record, indent=2) + "\n")
    sys.path.insert(0, str(REPO / "src"))
    SETUP_READY = True
    print("Source commit:", code_commit, "| source:", source_kind)
    print("python:", sys.version)
    print("Setup complete. Durable model revision:", pin["model_revision"])
    print("Acquisition and generation remain separate explicit phases.")
else:
    print("Setup disabled; no Drive mount, install, download, or GPU operation.")

## 2. Acquire and inventory once

Set `RUN_ACQUIRE=True` only to obtain the official pinned benchmark via its
documented procedure. Source records, mappings and canaries remain private. On
rerun, an existing inventory is validated and retained: opaque IDs and the split
are never regenerated just because a runtime restarted. Existing malformed or
changed data causes an error; no fixture replaces missing real data.

Human review of family grouping remains required before freeze. This cell prints
aggregate counts only. It does not display benchmark messages or annotations.

In [ ]:
if RUN_ACQUIRE:
    require_setup()
    from contextlib import chdir

    from context_audit.cli import main
    from context_audit.dataset import load_dataset

    with chdir(REPO):
        if not Path("data/private/manifest.json").exists():
            if main(["acquire"]):
                raise RuntimeError("Official dataset acquisition failed.")
            if main(["inventory"]):
                raise RuntimeError("Dataset inventory failed.")
        inputs, labels = load_dataset(Path("data/private"), "development")
        print("Validated development transcripts:", len(inputs))
        print("Existing opaque IDs and family split retained.")
        save_public_manifests()
else:
    print("Acquisition disabled; no real dataset is opened.")

## Optional recovery after an interrupted VM

An unclean VM/session interruption leaves its GPU-session reservation charged
against the phase cap. The supervisor will refuse automatic resumption while the
elapsed runtime is unknown; it never treats the lost receipt as zero cost.

After setup, inspect the private session metadata under
`runs/private/qwen-<phase>/gpu_sessions/` on Drive and determine the actual managed
elapsed seconds from billing or timestamp evidence. Set the matching `PHASE`,
`RECONCILE_SESSION_ID`, `RECONCILE_ELAPSED_SECONDS`, and the explicit cumulative
`MAX_COST_USD`. Set **both** `RUN_RECONCILE=True` and
`CONFIRM_RECONCILIATION=True` only after checking that evidence. This cell records
your confirmation and recomputes the saved cost receipt without starting a GPU.
Do not invent elapsed time or replace an uncertain receipt with zero.

Keep `RUN_LIVE=False` for recovery. Disable `RUN_RECONCILE` afterward and resume
that same phase with its original configuration and durable run directory.

In [ ]:
# SESSION_RECONCILIATION
if RUN_RECONCILE:
    require_setup()
    if not CONFIRM_RECONCILIATION:
        raise ValueError("Set CONFIRM_RECONCILIATION=True after checking elapsed-time evidence.")
    if RUN_LIVE or RUN_FREEZE:
        raise ValueError("Reconciliation must be separate from generation and freeze.")
    if PHASE not in {"pilot", "development", "test"}:
        raise ValueError("Choose the phase containing the interrupted session.")
    if not RECONCILE_SESSION_ID or RECONCILE_ELAPSED_SECONDS is None or MAX_COST_USD is None:
        raise ValueError("Supply session ID, verified elapsed seconds, and the explicit phase cap.")
    from contextlib import chdir

    from context_audit.colab import reconcile_gpu_session

    with chdir(REPO):
        reconcile_gpu_session(
            Path(f"runs/private/qwen-{PHASE}"),
            RECONCILE_SESSION_ID,
            elapsed_seconds=RECONCILE_ELAPSED_SECONDS,
            max_cost_usd=MAX_COST_USD,
            confirmed=True,
        )
    print("Verified interrupted-session accounting saved. Resume remains a separate explicit step.")
else:
    print("Session reconciliation disabled; no accounting records changed.")

## 3. Run one authorized phase, save, then release the runtime

Start with `PHASE="pilot"` (three development pairs). Fill the exact hourly rate and
phase cap; set data-use/rubric flags only after your review. Set `RUN_LIVE=True`.
All phases use a 300-second request timeout, allowing long prefill and summary
outputs on the 27B model. The helper owns the loopback server, model download/startup,
experiment and teardown.
It reserves runtime cost before launch and saves incremental accounting on Drive.
The cap is cumulative within this phase directory; separate phases need separately
allocated caps. Estimated GPU time cost is not a provider invoice.

Inspect private pilot evidence before proceeding. For the full development pass,
choose `PHASE="development"`; this evaluates all development pairs. These two runs
have distinct durable directories. A successful small pilot alone cannot freeze
the test. Changed methods need an explicit new workspace/version; preserve the
existing records for honest development provenance.

`PHASE="test"` additionally requires `RUN_TEST=True`, the reviewed freeze below,
and the unchanged tagged code/protocol. No phase automatically starts another.
`DISCONNECT_AFTER_RUN=True` releases Colab after the managed run and local numeric
export; set it false only if you accept the additional idle GPU allocation.

In [ ]:
# LIVE_EXECUTION
if RUN_LIVE:
    if RUN_RECONCILE:
        raise RuntimeError("Reconcile first, then disable RUN_RECONCILE before generation.")
    if RUN_FREEZE:
        raise RuntimeError("Freeze and generation are separate phases; disable RUN_FREEZE first.")
    if PHASE == "test" and not RUN_TEST:
        raise RuntimeError("Test execution requires the separate RUN_TEST=True opt-in.")
    require_setup()
    import json
    import math
    from contextlib import chdir

    if MAX_COST_USD is None or not math.isfinite(MAX_COST_USD) or MAX_COST_USD <= 0:
        raise ValueError("Provide an explicit positive finite MAX_COST_USD for this phase.")
    from context_audit.cli import export_results
    from context_audit.colab import run_colab_experiment
    from context_audit.reporting import generate_report

    config = configured_phase(PHASE)
    try:
        with chdir(REPO):
            summary = run_colab_experiment(
                config,
                max_cost_usd=MAX_COST_USD,
                session_max_seconds=SESSION_MAX_SECONDS,
                startup_timeout_seconds=STARTUP_TIMEOUT_SECONDS,
            )
            print(json.dumps(summary, indent=2))
    finally:
        # Preserve numeric partial results even after a managed timeout or failed run.
        try:
            with chdir(REPO):
                if (Path(config.run_dir) / "scores.csv").exists():
                    output = DRIVE_ROOT / "numeric-results" / PHASE
                    export_results(Path(config.run_dir), output)
                    generate_report(
                        output / "public_scores.csv",
                        output,
                        manifest_path=output / "run_manifest.json",
                    )
                    print("Numeric results and CPU analysis saved in the durable phase directory.")
        finally:
            # The helper persists session accounting before returning or raising.
            if DISCONNECT_AFTER_RUN:
                from google.colab import runtime

                runtime.unassign()
else:
    print("Live phase disabled; no model or GPU operation.")

## 4. Reviewed freeze: explicit local commit and tag

After the complete development run, review hypothesis, rubric, family grouping,
failures, exact counts, context fit, summaries and costs. Keep `RUN_LIVE=False` here.
Then set **both** `RUN_FREEZE=True` and `REVIEWED_FREEZE=True`. This cell verifies
completed matching development evidence before preparing the test config and freeze.
It creates a **local** protocol commit/tag and durable public-source snapshot; it
never pushes or publishes. Reconnect restores this frozen snapshot and Git metadata.

Any test-informed method change needs a documented exploratory protocol version,
not an overwritten `protocol-v1`. To run test later, rerun setup, set `PHASE="test"`,
`RUN_TEST=True` and `RUN_LIVE=True`, and allocate its own cap. Keep all other method
settings identical. The runner verifies the freeze again before scoring.

In [ ]:
if RUN_FREEZE:
    require_setup()
    if not REVIEWED_FREEZE:
        raise ValueError("Set REVIEWED_FREEZE=True only after reviewing completed development.")
    if RUN_LIVE:
        raise ValueError("Freeze is a separate step; set RUN_LIVE=False for this cell.")
    import json
    import subprocess
    from contextlib import chdir

    from context_audit.cli import freeze

    with chdir(REPO):
        test_config = configured_phase("test")
        result = freeze(test_config, Path("runs/private/qwen-development"))
        print(json.dumps(result, indent=2))
        tagged = subprocess.run(
            ["git", "rev-parse", "--verify", "protocol-v1^{commit}"],
            capture_output=True, text=True,
        )
        if tagged.returncode:
            public_paths = [
                "src", "tests", "prompts", "configs", "docs", "notebooks", "data/manifests",
                "pyproject.toml", "uv.lock", ".gitignore", "README.md", "AGENTS.md",
            ]
            subprocess.run(["git", "add", "--all", "--", *public_paths], check=True)
            subprocess.run(
                [
                    "git", "-c", "user.name=Colab protocol snapshot",
                    "-c", "user.email=local-colab@invalid", "commit",
                    "-m", "Freeze reviewed Qwen protocol-v1",
                ], check=True,
            )
            subprocess.run(["git", "tag", "protocol-v1"], check=True)
        else:
            head = subprocess.run(
                ["git", "rev-parse", "HEAD"], capture_output=True, text=True, check=True
            ).stdout.strip()
            if tagged.stdout.strip() != head:
                raise ValueError("Existing protocol-v1 does not identify HEAD; no tag was changed.")
        archive = DRIVE_ROOT / "frozen-source.zip"
        subprocess.run(
            ["git", "archive", "--format=zip", f"--output={archive}", "HEAD"], check=True
        )
        save_public_manifests()
        print("Reviewed local freeze and source snapshot saved. Test remains a separate opt-in.")
else:
    print("Freeze disabled; no Git commit or tag is created.")

## 5. Offline reproduction from saved numeric results

Set `RUN_ANALYSIS=True` after setup to recalculate an exported phase without GPU
or generation. The authoritative run manifest supplies frozen bootstrap settings
and planned coverage. No scores means **experiment not executed**; no plausible
numbers are filled in. Notebook 02 offers a richer view of these same numeric
artifacts. Review text and figures before any separately authorized publication.

Do not place private justifications, source examples or completed qualitative
worksheets in this notebook. Keep committed/shared notebook outputs empty.

In [ ]:
if RUN_ANALYSIS:
    require_setup()
    from context_audit.reporting import generate_report

    numeric = DRIVE_ROOT / "numeric-results" / PHASE
    manifest = numeric / "run_manifest.json"
    metrics = generate_report(
        numeric / "public_scores.csv",
        numeric / "reproduced",
        manifest_path=manifest if manifest.exists() else None,
    )
    print("Offline analysis status:", metrics["status"])
    print("Results are saved on Drive; this cell makes no model call.")
else:
    print("Analysis disabled; no result files are read or written.")